# Phase 3 - Step 6: ensemble (extra submissions) + ablations for the analysis

**Part A: ensemble (no new training).** Every run saved its predicted probabilities for every dev and test
sentence (`dev_logprobs.npz`, `test_logprobs.npz`). An ensemble averages these probabilities over several runs
and takes the most likely role. Different runs make different mistakes (their test predictions disagree on
15-20% of the sentences), so averaging cancels part of them.

- **Ensemble-4** = the four 10-layer decoder runs (#1 A2 full, #2 B LoRMA, #3 A LoRA, #5 A2 IA³). No selection on
  dev: picking the "best subset" on 10 dev documents would overfit the dev set.
- **Ensemble-5** = Ensemble-4 + #6 (encoder + rsLoRA), a different architecture (bidirectional encoder).
  Dev scores use the Stage-A models (trained on train only); test predictions use the Stage-B models.

**Part B: logit adjustment (analysis).** Subtract tau \* log(class prior) from the scores to favour rare roles
(Menon et al. 2021). Measured on dev for tau = 0 ... 1.

**Part C: ablations on the best model (A + LoRA), dev only, ~15 min each.** The doc asks whether context helps:

- sentence-only: every sentence read alone, no BiLSTM, no position (no document context at all)
- no position: the full model without the relative-position feature (my own design choice D)


In [1]:
import sys, json
from pathlib import Path
import numpy as np
import torch

ROOT = Path.cwd().resolve()
while not (ROOT / ".gitignore").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Project root with .gitignore not found")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import importlib, rr_utils, rr_models, rr_experiment
for m in (rr_utils, rr_models, rr_experiment):
    importlib.reload(m)
from rr_experiment import Experiment, compare
import rr_utils as ru

LABELS, DATA = ru.load_rr_data(ROOT)
CK = ROOT / "checkpoints" / "phase3"
OUT = ROOT / "experiments" / "phase3"
DECODERS = ["A2_full_causal_doc", "B_lormar8_causal_doc", "A_lorar8_causal_doc", "A2_ia3_causal_doc"]
ENSEMBLES = {"ensemble4": DECODERS, "ensemble5": DECODERS + ["D_rslorar32_bidir_doc_w1024"]}

def load_lp(run, split):
    return dict(np.load(CK / run / f"{split}_logprobs.npz"))

def ensemble(runs, split):
    """Average the PROBABILITIES of the runs for every sentence, return log-probabilities."""
    lps = [load_lp(r, split) for r in runs]
    return {k: np.log(np.mean([np.exp(lp[k]) for lp in lps], axis=0) + 1e-12) for k in lps[0]}

def to_labels(lp):
    return {k: [LABELS[i] for i in v.argmax(1)] for k, v in lp.items()}

c:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## A. Ensembles: dev scores and test submissions


In [2]:
dev_rows = {}
for r in DECODERS + ["D_rslorar32_bidir_doc_w1024", "D_kd_bidir_doc_w1024"]:
    dev_rows[r] = ru.score_by_domain(DATA["dev"], to_labels(load_lp(r, "dev")), LABELS)
for name, runs in ENSEMBLES.items():
    dev_rows[name] = ru.score_by_domain(DATA["dev"], to_labels(ensemble(runs, "dev")), LABELS)
print(f"{'model':32s} {'dev macro-F1':>12s} {'CL':>7s} {'IT':>7s} {'accuracy':>9s}")
for name, sc in dev_rows.items():
    print(f"{name:32s} {sc['all']['macro_f1']:12.4f} {sc['CL']['macro_f1']:7.4f} {sc['IT']['macro_f1']:7.4f} {sc['all']['accuracy']:9.4f}")
ru.print_rr_scores(dev_rows["ensemble4"], "\nEnsemble-4 on dev, per label:")

model                            dev macro-F1      CL      IT  accuracy
A2_full_causal_doc                     0.5616  0.4883  0.5434    0.6228
B_lormar8_causal_doc                   0.5727  0.5388  0.5286    0.6433
A_lorar8_causal_doc                    0.5838  0.4885  0.5939    0.6232
A2_ia3_causal_doc                      0.5660  0.5159  0.5572    0.6237
D_rslorar32_bidir_doc_w1024            0.5299  0.4614  0.5090    0.5924
D_kd_bidir_doc_w1024                   0.4444  0.3845  0.4186    0.5341
ensemble4                              0.5985  0.5137  0.5916    0.6489
ensemble5                              0.5940  0.5118  0.5801    0.6480

Ensemble-4 on dev, per label:
  all: macro-F1 0.5985 | macro-P 0.6132 | macro-R 0.6055 | accuracy 0.6489 | macro-F1 over all 13 0.5064 | 11 labels, 2142 sentences
  CL : macro-F1 0.5137 | macro-P 0.5109 | macro-R 0.5419 | accuracy 0.6583 | macro-F1 over all 13 0.4346 | 11 labels, 1396 sentences
  IT : macro-F1 0.5916 | macro-P 0.6658 | macro-R 0.570

In [3]:
for name, runs in ENSEMBLES.items():
    sub = OUT / name / "submission"
    if (sub / "submission.zip").exists():
        ru.check_submission(sub / "submission.zip", DATA["test"], LABELS)
        print(f"[cache] {name} submission already written")
        continue
    lp = ensemble(runs, "test")
    (OUT / name).mkdir(parents=True, exist_ok=True)
    np.savez_compressed(OUT / name / "test_logprobs.npz", **lp)
    ru.write_submission(to_labels(lp), DATA["test"], LABELS, sub)
json.dump({k: {d: v[d]["macro_f1"] for d in v} for k, v in dev_rows.items()}, open(OUT / "p3_06_ensemble_dev.json", "w"), indent=1)

submission OK: C:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\experiments\phase3\ensemble4\submission\submission.zip | 10 documents | 2,197 sentences | label counts {'RatioOfTheDecision': 769, 'Fact': 422, 'PrecedentReliedUpon': 310, 'ArgumentPetitioner': 222, 'Statute': 162, 'RulingByLowerCourt': 108, 'ArgumentRespondent': 102, 'RulingByPresentCourt': 43, 'None': 24, 'Issue': 15, 'PrecedentNotReliedUpon': 14, 'Dissent': 6}
submission OK: C:\Users\Atharva\Desktop\Course Sem 1\LLM\Capstone Project\experiments\phase3\ensemble5\submission\submission.zip | 10 documents | 2,197 sentences | label counts {'RatioOfTheDecision': 768, 'Fact': 420, 'PrecedentReliedUpon': 321, 'ArgumentPetitioner': 217, 'Statute': 163, 'RulingByLowerCourt': 106, 'ArgumentRespondent': 105, 'RulingByPresentCourt': 43, 'None': 24, 'Issue': 15, 'PrecedentNotReliedUpon': 14, 'Dissent': 1}


## B. Logit adjustment (analysis only)

score(role) = log p(role) - tau \* log prior(role). tau = 0 is the normal prediction; larger tau pushes the
prediction towards rare roles (more recall for them, less precision).


In [4]:
counts = np.bincount([LABELS.index(l) for d in DATA["train"] for l in d["labels"]], minlength=len(LABELS)) + 1
log_prior = np.log(counts / counts.sum())
best_single, ens4 = load_lp("A_lorar8_causal_doc", "dev"), ensemble(DECODERS, "dev")
la = {}
for tau in (0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.75, 1.0):
    la[tau] = {name: ru.score_by_domain(DATA["dev"], to_labels({k: v - tau * log_prior for k, v in lp.items()}), LABELS)["all"]["macro_f1"]
               for name, lp in (("A + LoRA", best_single), ("ensemble4", ens4))}
    print(f"tau {tau:4.2f}: " + " | ".join(f"{n} {s:.4f}" for n, s in la[tau].items()))
json.dump(la, open(OUT / "p3_06_logit_adjustment.json", "w"), indent=1)

tau 0.00: A + LoRA 0.5838 | ensemble4 0.5985
tau 0.10: A + LoRA 0.5849 | ensemble4 0.5949
tau 0.20: A + LoRA 0.5824 | ensemble4 0.5928
tau 0.30: A + LoRA 0.5860 | ensemble4 0.5902
tau 0.40: A + LoRA 0.5842 | ensemble4 0.5916
tau 0.50: A + LoRA 0.5804 | ensemble4 0.5378
tau 0.75: A + LoRA 0.5291 | ensemble4 0.5179
tau 1.00: A + LoRA 0.5232 | ensemble4 0.5024


## C. Ablations on A + LoRA (dev only)

Same settings as experiment #3 except the one thing removed. Compare with A + LoRA dev macro-F1 0.5838.


In [5]:
ABLATIONS = {
    "sentence-only (no context)": {"phase1": "A", "method": "lora", "rank": 8, "alpha": 16, "context": "sentence"},
    "no position feature": {"phase1": "A", "method": "lora", "rank": 8, "alpha": 16, "use_pos": False},
}
for name, ov in ABLATIONS.items():
    print(f"\n===== {name}")
    exp = Experiment(ROOT, ov)
    exp.stage_a()
    exp.report(show_per_label=False)
    exp.log()


===== sentence-only (no context)


experiment: A_lorar8_causal_sentence | device cuda | precision torch.bfloat16
parameters: 71.27M total, 1.219M trainable | loaded {'arch': 'A_decoder', 'layers': 10, 'loaded_tensors': 72, 'dropped_tensors': 1, 'dropped_groups': ['lm_head'], 'adapter_params': 875520}


epoch  1 | train loss 2.2955 | 102s | dev macro-F1 0.1890 (CL 0.1835, IT 0.2044) acc 0.3315  <- best


epoch  2 | train loss 1.7207 | 98s | dev macro-F1 0.3696 (CL 0.2916, IT 0.4121) acc 0.3996  <- best


epoch  3 | train loss 1.4996 | 100s | dev macro-F1 0.3750 (CL 0.2795, IT 0.4258) acc 0.4248  <- best


epoch  4 | train loss 1.3522 | 100s | dev macro-F1 0.3914 (CL 0.2884, IT 0.4605) acc 0.4234  <- best


... (21 outputs left out)

epoch 16 | train loss 0.0522 | 69s | dev macro-F1 0.5840 (CL 0.5551, IT 0.5487) acc 0.6354


epoch 17 | train loss 0.0434 | 68s | dev macro-F1 0.5656 (CL 0.4904, IT 0.5399) acc 0.6275


epoch 18 | train loss 0.0383 | 67s | dev macro-F1 0.5501 (CL 0.4528, IT 0.5304) acc 0.6326
early stop: no improvement for 5 epochs
training 'train' finished -> loaded best epoch 13
Stage A: best epoch 13 | dev macro-F1 0.5855 | 1049s | peak GPU 1.46 GB


A_lorar8_causal_doc_nopos - dev:
  all: macro-F1 0.5855 | macro-P 0.6143 | macro-R 0.5827 | accuracy 0.6466 | macro-F1 over all 13 0.4954 | 11 labels, 2142 sentences
  CL : macro-F1 0.5256 | macro-P 0.5807 | macro-R 0.5353 | accuracy 0.6676 | macro-F1 over all 13 0.4448 | 11 labels, 1396 sentences
  IT : macro-F1 0.5552 | macro-P 0.6468 | macro-R 0.5368 | accuracy 0.6072 | macro-F1 over all 13 0.4698 | 11 labels, 746 sentences
most confused (gold -> predicted): [('ArgumentRespondent -> ArgumentPetitioner', 111), ('PrecedentReliedUpon -> RatioOfTheDecision', 61), ('ArgumentPetitioner -> RatioOfTheDecision', 52), ('Fact -> RatioOfTheDecision', 49), ('PrecedentReliedUpon -> Fact', 42), ('ArgumentPetitioner -> ArgumentRespondent', 36), ('Fact -> RulingByLowerCourt', 28), ('Fact -> ArgumentPetitioner', 25)]
logged: A_lorar8_causal_doc_nopos | dev macro-F1 0.5855


In [6]:
compare(ROOT)

run                                     macro     CL     IT    acc  ep     Fact   Issue Argumen Argumen Statute Dissent Precede Precede Precede RulingB RatioOf RulingB    None
A_lorar8_causal_doc_nopos              0.5855 0.5256 0.5552 0.6466  13    0.741   0.690   0.633   0.496   0.787   0.000   0.567   0.358   0.000   0.537   0.719   0.566   0.346
A_lorar8_causal_doc                    0.5838 0.4885 0.5939 0.6232  12    0.744   0.710   0.392   0.549   0.812   0.000   0.577   0.394   0.000   0.616   0.731   0.551   0.346
B_lormar8_causal_doc                   0.5727 0.5388 0.5286 0.6433   7    0.753   0.545   0.599   0.560   0.824   0.000   0.634   0.336   0.000   0.450   0.698   0.600   0.300
A2_ia3_causal_doc                      0.5660 0.5159 0.5572 0.6237  12    0.722   0.538   0.488   0.614   0.750   0.000   0.557   0.312   0.000   0.667   0.678   0.592   0.308
A2_full_causal_doc                     0.5616 0.4883 0.5434 0.6228   7    0.715   0.629   0.574   0.561   0.700   0.000 

[('A_lorar8_causal_doc_nopos',
  0.5854844736917022,
  0.5256203056957991,
  0.5552269239823863,
  0.6465919701213819,
  13,
  {'Fact': 0.7414,
   'Issue': 0.6897,
   'ArgumentPetitioner': 0.6331,
   'ArgumentRespondent': 0.4964,
   'Statute': 0.7871,
   'Dissent': 0.0,
   'PrecedentReliedUpon': 0.5671,
   'PrecedentNotReliedUpon': 0.3577,
   'PrecedentOverruled': 0.0,
   'RulingByLowerCourt': 0.5368,
   'RatioOfTheDecision': 0.7192,
   'RulingByPresentCourt': 0.5657,
   'None': 0.3462}),
 ('A_lorar8_causal_doc',
  0.583777731396533,
  0.4884868641898356,
  0.5939389340521662,
  0.623249299719888,
  12,
  {'Fact': 0.7441,
   'Issue': 0.7097,
   'ArgumentPetitioner': 0.3915,
   'ArgumentRespondent': 0.549,
   'Statute': 0.8117,
   'Dissent': 0.0,
   'PrecedentReliedUpon': 0.5773,
   'PrecedentNotReliedUpon': 0.3944,
   'PrecedentOverruled': 0.0,
   'RulingByLowerCourt': 0.6162,
   'RatioOfTheDecision': 0.7306,
   'RulingByPresentCourt': 0.551,
   'None': 0.3462}),
 ('B_lormar8_causal_do